# IT8532 Assessment 2: Unsloth QLoRA Fine-Tuning & GGUF Export
**Base Model:** `Qwen/Qwen2.5-3B-Instruct`  
**Quantization:** 4-bit  
**Export:** GGUF `Q4_K_M` (`qwen2.5-3b-instruct-q4_k_m.gguf`)

In [ ]:
# Step 1: Install Unsloth & Fine-Tuning Dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

In [ ]:
# Step 2: Load Qwen2.5-3B-Instruct Base Model in 4-bit
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# Add QLoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
)

In [ ]:
# Step 3: Load & Format Conversations Dataset
from datasets import load_dataset

dataset = load_dataset("json", data_files="finetune_dataset.json", split="train")

def formatting_prompts_func(examples):
    convs = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convs]
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"Successfully formatted {len(dataset)} instruction examples with ChatML template.")

In [ ]:
# Step 4: Fine-Tune with SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        num_train_epochs = 1,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        output_dir = "outputs",
    ),
)
trainer_stats = trainer.train()

In [ ]:
# Step 5: Test Model Inference & JSON Schema Accuracy Verification
import json

FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "You are a task extraction assistant for a time management system. Extract scheduling information from the user's natural language task description and return ONLY a valid JSON object with exactly these fields: task, hours_to_deadline, est_duration_h, urgency, importance, has_dependency, dependency_note."},
    {"role": "user", "content": "Complete the client financial report by tonight. Takes roughly 3 hours. It is critical importance 5 out of 5."}
]

inputs = tokenizer.apply_chat_template(messages, tokenize = True, add_generation_prompt = True, return_tensors = "pt").to("cuda")
outputs = model.generate(input_ids = inputs, max_new_tokens = 256, use_cache = True)
response_text = tokenizer.batch_decode(outputs)[0]

if "<|im_start|>assistant" in response_text:
    response_body = response_text.split("<|im_start|>assistant")[-1].replace("<|im_end|>", "").strip()
else:
    response_body = response_text.strip()

print("==========================================================")
print("MODEL INFERENCE OUTPUT:")
print(response_body)
print("==========================================================")

# Verify JSON Accuracy & Schema Keys
try:
    start_idx = response_body.find("{")
    end_idx = response_body.rfind("}") + 1
    if start_idx != -1 and end_idx != 0:
        json_str = response_body[start_idx:end_idx]
        extracted_json = json.loads(json_str)
        required_keys = ["task", "hours_to_deadline", "est_duration_h", "urgency", "importance", "has_dependency", "dependency_note"]
        valid_keys = all(k in extracted_json for k in required_keys)
        print("JSON Syntax Check        : PASS (Valid JSON)")
        print(f"Schema Compliance Check : {'PASS (100% Keys Present)' if valid_keys else 'FAIL'}")
    else:
        print("JSON Syntax Check        : FAIL (No JSON brackets found)")
except Exception as e:
    print(f"JSON Syntax Check        : FAIL ({e})")

In [ ]:
# Step 6: Export Merged Model to GGUF (Q4_K_M)
model.save_pretrained_gguf("qwen_gguf", tokenizer, quantization_method = "q4_k_m")
print("Exported fine-tuned model to GGUF Q4_K_M successfully.")